# ARC-AGI-2 — baseline submission

**This is a participation entry, not a competitive one.** It is a program-synthesis baseline over a small library of grid transformations, built to satisfy the ARC Prize 2026 Paper Track's requirement that the team make a submission to ARC-AGI-2 or ARC-AGI-3. Measured score on the evaluation set: **0.0%**, with a non-zero control on the easier ARC-AGI-1 training set (9.8%) confirming the solver and the scorer work.

ARC-AGI-2 is constructed to defeat exactly this class of fixed-library approach. The notebook reports that rather than dressing it up.

## Solver

Embedded verbatim from `tools/arc_baseline.py` at build time, so the implementation that produced the recorded scores is the one that runs here.

In [ ]:
#!/usr/bin/env python3
"""ARC-AGI-2 baseline solver and submission writer.

★ WHY THIS EXISTS AT ALL. The ARC Prize 2026 Paper Track requires the team to have made a
submission to ARC-AGI-2 or ARC-AGI-3. That requirement is about participation, not score, so a
legitimate minimal solver is what keeps the door open — and it must be honest about being minimal.

★ WHAT THIS IS AND IS NOT. It is a program-synthesis baseline over a small library of grid
transformations: it searches for a transformation consistent with every training pair of a task,
then applies it to the test inputs. It is NOT a learned system, NOT an LLM wrapper, and NOT a claim
about ARC capability. Reported scores from it should be read as a floor, and it will score near zero
on the 2026 set, which is designed to defeat exactly this class of approach.

★ THE FORMAT IS THE PART THAT MATTERS. A submission is JSON mapping each task id to a list of
attempts, each attempt a grid (list of rows, each a list of ints). ARC-AGI-2 allows two attempts
per test input, so a solver that is unsure can offer two candidates instead of one. Producing a
well-formed file is the whole requirement here; guessing badly is worse than guessing once.

"""
import argparse
import itertools
import json
import sys
from collections import Counter

Grid = list  # list[list[int]]


# ─── the transformation library ──────────────────────────────────────────────────────────────────
# Each candidate is a pure function Grid -> Grid, applied uniformly to every pair. A candidate is
# accepted only if it maps EVERY training input to its exact training output; a transformation that
# fits some pairs and not others is not a hypothesis, it is a coincidence.

def _rot90(g):
    return [list(r) for r in zip(*g[::-1])]


def _flip_h(g):
    return [r[::-1] for r in g]


def _flip_v(g):
    return g[::-1]


def _transpose(g):
    return [list(r) for r in zip(*g)]


def _identity(g):
    return [list(r) for r in g]


def _rotations():
    return [_identity, _rot90, lambda g: _rot90(_rot90(g)), lambda g: _rot90(_rot90(_rot90(g)))]


def _flips():
    return [_identity, _flip_h, _flip_v, lambda g: _flip_v(_flip_h(g)), _transpose,
            lambda g: _rot90(_transpose(g)), lambda g: _rot90(_rot90(_transpose(g))),
            lambda g: _rot90(_rot90(_rot90(_transpose(g))))]


def _constant_color_map(pairs):
    """Map colours consistently: the modal (in -> out) colour correspondence across all pairs."""
    mapping = {}
    for gi, go in pairs:
        if len(gi) != len(go) or len(gi[0]) != len(go[0]):
            return None
        for ri, ro in zip(gi, go):
            for a, b in zip(ri, ro):
                if a in mapping and mapping[a] != b:
                    return None
                mapping[a] = b
    if not mapping:
        return None

    def f(g):
        return [[mapping.get(v, v) for v in row] for row in g]
    return f


def _outline_of_largest_object(pairs):
    """Crop to the bounding box of the largest non-background object, a common ARC motif."""
    def bbox(g):
        h, w = len(g), len(g[0])
        bg = Counter(v for r in g for v in r).most_common(1)[0][0]
        cells = [(r, c) for r in range(h) for c in range(w) if g[r][c] != bg]
        if not cells:
            return None
        r0 = min(r for r, _ in cells); r1 = max(r for r, _ in cells)
        c0 = min(c for _, c in cells); c1 = max(c for _, c in cells)
        return [row[c0:c1 + 1] for row in g[r0:r1 + 1]]
    return bbox


def _tile_to(g, kh, kw):
    """Tile the grid k times vertically and horizontally."""
    out = []
    for _ in range(kh):
        for row in g:
            out.append(row * kw)
    return out


def _tile_candidates(pairs):
    """Infer a tile factor from the first pair and verify on the rest."""
    (gi0, go0) = pairs[0]
    if not gi0 or not gi0[0]:
        return
    kh = len(go0) // len(gi0)
    kw = len(go0[0]) // len(gi0[0])
    if kh > 1 or kw > 1:
        yield lambda g, kh=kh, kw=kw: _tile_to(g, kh, kw)


def solve_task(task):
    """Return up to 2 candidate grids for each test input, best-first.

    Args:
        task: {"train": [{"input": Grid, "output": Grid}, ...], "test": [{"input": Grid}, ...]}
    """
    train = task.get("train") or []
    test = task.get("test") or []
    pairs = [(t["input"], t["output"]) for t in train if "input" in t and "output" in t]
    candidates = []

    # 1. geometric transforms, verified against every training pair
    for name, fn in zip(
            ["id", "rot", "rot180", "rot270", "fh", "fv", "fhfv", "tr", "tr_rot", "tr_r2", "tr_r3"],
            list(itertools.islice(itertools.chain(_rotations()), 4)) + _flips()[1:]):
        try:
            if all(fn(gi) == go for gi, go in pairs):
                candidates.append((name, fn))
        except Exception:
            continue

    # 2. consistent colour mapping
    cm = _constant_color_map(pairs)
    if cm is not None and all(cm(gi) == go for gi, go in pairs):
        candidates.append(("color_map", cm))

    # 3. tile factors
    if pairs:
        try:
            for tf in _tile_candidates(pairs):
                if all(tf(gi) == go for gi, go in pairs):
                    candidates.append(("tile", tf))
        except Exception:
            pass

    # 4. bounding box of the largest object
    try:
        bb = _outline_of_largest_object(pairs)
        if all(bb(gi) == go for gi, go in pairs):
            candidates.append(("bbox", bb))
    except Exception:
        pass

    # Build the attempts. With no verified transformation, fall back to identity -- offering a
    # well-formed guess costs nothing and a malformed file is a rejected submission.
    attempts = []
    seen = set()
    for test_case in test:
        ti = test_case.get("input")
        opts = []
        for _, fn in candidates:
            try:
                g = fn(ti)
            except Exception:
                continue
            key = json.dumps(g)
            if key not in seen:
                opts.append(g)
                seen.add(key)
            if len(opts) == 2:
                break
        while len(opts) < 2:
            opts.append([list(r) for r in ti])       # identity as the fill attempt
        attempts.append(opts)
    return attempts, [c[0] for c in candidates]

## Run

The competition is kernels-only, so this cell is the submission: it reads the test challenges and writes `submission.json`.

In [ ]:
# ─── Kaggle driver ────────────────────────────────────────────────────────────────────────────
# The competition is KERNELS-ONLY, so this cell is what actually constitutes a submission: it runs
# inside Kaggle, reads the test challenges, and writes submission.json to the working directory.
import glob, json, os, sys

# The mounted input directory has varied across ARC releases, so it is discovered rather than
# hardcoded -- a wrong hardcoded path produces an empty submission that still "runs successfully".
CANDIDATES = [
    "/kaggle/input/arc-prize-2026-arc-agi-2",
    "/kaggle/input/arc-prize-2026-arc-agi-2/",
]
INPUT = None
for c in CANDIDATES:
    if os.path.isdir(c):
        INPUT = c
        break
if INPUT is None:
    hits = glob.glob("/kaggle/input/*arc*")
    INPUT = hits[0] if hits else None
print("input dir:", INPUT)

# Find the challenge file: the test set is JSON with a "test" key and no "output".
challenge_path = None
if INPUT:
    for p in sorted(glob.glob(os.path.join(INPUT, "**", "*.json"), recursive=True)):
        try:
            d = json.load(open(p))
        except Exception:
            continue
        if isinstance(d, dict) and d and all(isinstance(v, dict) and "test" in v for v in d.values()):
            challenge_path = p
            break
print("challenges:", challenge_path)

if challenge_path is None:
    # Fail LOUDLY. An empty submission that exits 0 is indistinguishable from a solved one at the
    # result level, which is the failure this whole repository is about.
    raise SystemExit("no challenge file found under %r -- refusing to write an empty submission" % INPUT)

tasks = json.load(open(challenge_path))
print("tasks:", len(tasks))

submission = {}
solved = 0
for tid, task in tasks.items():
    attempts, names = solve_task(task)
    if names:
        solved += 1
    submission[tid] = attempts

out = "/kaggle/working/submission.json"
with open(out, "w") as fh:
    json.dump(submission, fh)
print("wrote %s  (%d tasks, %d with a verified hypothesis)" % (out, len(submission), solved))

# Format self-check before finishing: the grader rejects a malformed file, and a rejected file
# scores nothing regardless of the solver. Check the shape, do not assume it.
# ★ THE CONTRACT, taken from the competition's own sample_submission.json rather than assumed:
#   {"<task>": [{"attempt_1": grid, "attempt_2": grid}, ...]}  -- one dict per test input.
# An earlier version checked for a list of two grids, which is self-consistent and wrong; a format
# check written from an assumption validates the assumption, not the contract.
bad = []
for tid, preds in submission.items():
    if not isinstance(preds, list) or not preds:
        bad.append((tid, "no predictions")); continue
    for entry in preds:
        if not isinstance(entry, dict) or set(entry) != {"attempt_1", "attempt_2"}:
            bad.append((tid, "entry is not {attempt_1,attempt_2}")); break
        for key in ("attempt_1", "attempt_2"):
            g = entry[key]
            if not isinstance(g, list) or not g or not all(isinstance(r, list) for r in g):
                bad.append((tid, "%s is not a grid" % key)); break
print("format check:", "PASS" if not bad else "FAIL %r" % bad[:5])
print("submission bytes:", os.path.getsize(out))